# 🔍 Credit Card Fraud Detection — Visualization Notebook

**Dataset:** Sparkov Credit Card Fraud (Kaggle — kartik2112)  
**Models:** Logistic Regression · Decision Tree · Random Forest  

This notebook generates 10 visualizations:  
1. Class Distribution  
2. Transaction Amount Distribution  
3. Fraud Rate by Time (Hour & Day of Week)  
4. Fraud Rate by Merchant Category  
5. Feature Correlation Heatmap  
6. Feature Importance (Best Model)  
7. Confusion Matrix  
8. ROC Curve  
9. Precision-Recall Curve + Threshold Analysis  
10. Fraud Probability Distribution  

> **Run all cells top-to-bottom.** Make sure `train.py` has been run first so `models/best_model.pkl` exists.

## 1 · Imports & Setup

In [ ]:
import os
import warnings
import joblib

import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')

BASE_DIR  = os.getcwd()
DATA_DIR  = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
OUT_DIR   = os.path.join(BASE_DIR, 'outputs', 'visualizations')
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_CSV  = os.path.join(DATA_DIR, 'fraudTrain.csv')
TEST_CSV   = os.path.join(DATA_DIR, 'fraudTest.csv')
MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pkl')

CATEGORICAL_COLS = ['category', 'gender', 'state', 'job']
DROP_COLS = [
    'Unnamed: 0', 'trans_num', 'first', 'last',
    'street', 'city', 'zip', 'dob', 'merchant', 'cc_num',
]

PALETTE_FRAUD  = '#ef4444'
PALETTE_LEGIT  = '#3b82f6'
PALETTE_ACCENT = '#f59e0b'
PALETTE_BG     = '#f8fafc'

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.facecolor': PALETTE_BG,
    'axes.facecolor':   'white',
    'axes.edgecolor':   '#cbd5e1',
    'grid.color':       '#e2e8f0',
    'grid.linewidth':   0.6,
})

def savefig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=PALETTE_BG)
    print(f'  ✔ Saved → outputs/visualizations/{name}')

print('✅ Imports complete.')

ModuleNotFoundError: No module named 'joblib'

## 2 · Load Data & Engineer Features

In [ ]:
print('Loading datasets...')
raw_train = pd.read_csv(TRAIN_CSV, parse_dates=['trans_date_trans_time'])
raw_test  = pd.read_csv(TEST_CSV,  parse_dates=['trans_date_trans_time'])

print(f'Train rows : {len(raw_train):,}')
print(f'Test  rows : {len(raw_test):,}')
print(f'Columns    : {list(raw_train.columns)}')
raw_train.head(3)

In [ ]:
def engineer(df, encode_cats=True):
    df = df.copy()
    df['hour']       = df['trans_date_trans_time'].dt.hour
    df['dayofweek']  = df['trans_date_trans_time'].dt.dayofweek
    df['month']      = df['trans_date_trans_time'].dt.month
    df['is_night']   = df['hour'].between(0, 5).astype(int)
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['lat_diff']   = (df['lat'] - df['merch_lat']).abs()
    df['long_diff']  = (df['long'] - df['merch_long']).abs()
    df['geo_dist']   = np.sqrt(df['lat_diff']**2 + df['long_diff']**2)
    df['log_amt']    = np.log1p(df['amt'])
    dob_parsed       = pd.to_datetime(df['dob'], errors='coerce')
    df['age']        = ((df['trans_date_trans_time'] - dob_parsed).dt.days // 365)
    if encode_cats:
        for col in CATEGORICAL_COLS:
            if col in df.columns:
                df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    cols_to_drop = [c for c in DROP_COLS + [
        'trans_date_trans_time', 'lat', 'long',
        'merch_lat', 'merch_long', 'lat_diff', 'long_diff',
    ] if c in df.columns]
    df.drop(columns=cols_to_drop, inplace=True)
    df.dropna(inplace=True)
    return df

train_eng = engineer(raw_train)
test_eng  = engineer(raw_test)

X_train = train_eng.drop(columns=['is_fraud'])
y_train = train_eng['is_fraud']
X_test  = test_eng.drop(columns=['is_fraud'])
y_test  = test_eng['is_fraud']

print(f'Train shape : {X_train.shape}  | Fraud rate: {y_train.mean()*100:.2f}%')
print(f'Test  shape : {X_test.shape}   | Fraud rate: {y_test.mean()*100:.2f}%')
print('✅ Feature engineering complete.')

## 3 · Load Saved Model & Run Predictions

In [ ]:
assert os.path.exists(MODEL_PATH), (
    f'Model not found at {MODEL_PATH}.\n'
    'Please run  python train.py  first.'
)

bundle        = joblib.load(MODEL_PATH)
model         = bundle['model']
feature_names = bundle['feature_names']

X_test_aligned = X_test.reindex(columns=feature_names, fill_value=0)

y_pred = model.predict(X_test_aligned)
y_prob = model.predict_proba(X_test_aligned)[:, 1]

print(f'Model type    : {type(model).__name__}')
print(f'Features      : {feature_names}')
print(f'Test samples  : {len(y_test):,}')
print(f'Fraud flagged : {y_pred.sum():,}  ({y_pred.mean()*100:.2f}%)')
print(f'ROC-AUC       : {roc_auc_score(y_test, y_prob):.4f}')
print(f'PR-AUC        : {average_precision_score(y_test, y_prob):.4f}')
print('✅ Predictions complete.')

## Plot 1 · Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Class Distribution  (Fraud vs Legitimate)', fontsize=14, fontweight='bold', y=1.02)

for ax, (y, title) in zip(axes, [(y_train, 'Training Set'), (y_test, 'Test Set')]):
    counts = y.value_counts().sort_index()
    bars = ax.bar(['Legitimate', 'Fraud'], counts.values,
                  color=[PALETTE_LEGIT, PALETTE_FRAUD],
                  edgecolor='white', linewidth=1.5, width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + counts.max()*0.01,
                f'{val:,}\n({val/counts.sum()*100:.2f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Number of Transactions')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.set_ylim(0, counts.max() * 1.2)
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
savefig(fig, '01_class_distribution.png')
plt.show()

## Plot 2 · Transaction Amount Distribution

In [ ]:
fraud = raw_test[raw_test['is_fraud'] == 1]['amt']
legit = raw_test[raw_test['is_fraud'] == 0]['amt']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Transaction Amount: Fraud vs Legitimate', fontsize=14, fontweight='bold')

# KDE
ax = axes[0]
fraud.clip(upper=fraud.quantile(0.995)).plot.kde(ax=ax, color=PALETTE_FRAUD, linewidth=2, label='Fraud')
legit.clip(upper=legit.quantile(0.995)).plot.kde(ax=ax, color=PALETTE_LEGIT, linewidth=2, label='Legit')
ax.set_title('Amount Density (trimmed at 99.5th pct)')
ax.set_xlabel('Transaction Amount ($)')
ax.set_ylabel('Density')
ax.legend()

# Boxplot
ax2 = axes[1]
bp = ax2.boxplot(
    [legit.clip(upper=legit.quantile(0.99)).values,
     fraud.clip(upper=fraud.quantile(0.99)).values],
    patch_artist=True, notch=True,
    labels=['Legitimate', 'Fraud'],
    medianprops={'color': 'white', 'linewidth': 2}
)
for patch, color in zip(bp['boxes'], [PALETTE_LEGIT, PALETTE_FRAUD]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax2.set_title('Amount Boxplot (trimmed at 99th pct)')
ax2.set_ylabel('Transaction Amount ($)')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
savefig(fig, '02_amount_distribution.png')
plt.show()

## Plot 3 · Fraud Rate by Time of Day & Day of Week

In [ ]:
hour_rate = raw_train.assign(
    hour=raw_train['trans_date_trans_time'].dt.hour
).groupby('hour')['is_fraud'].mean() * 100

dow_rate = raw_train.assign(
    dow=raw_train['trans_date_trans_time'].dt.dayofweek
).groupby('dow')['is_fraud'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fraud Rate by Time', fontsize=14, fontweight='bold')

axes[0].bar(hour_rate.index, hour_rate.values, color=PALETTE_FRAUD, alpha=0.85, edgecolor='white')
axes[0].set_title('Fraud Rate by Hour of Day')
axes[0].set_xlabel('Hour of Day (0–23)')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xticks(range(0, 24, 2))

axes[1].bar(dow_rate.index, dow_rate.values, color=PALETTE_ACCENT, alpha=0.85, edgecolor='white')
axes[1].set_title('Fraud Rate by Day of Week')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
savefig(fig, '03_fraud_by_time.png')
plt.show()

## Plot 4 · Fraud Rate by Merchant Category

In [ ]:
cat_stats = (
    raw_train.groupby('category')['is_fraud']
    .agg(['sum','count'])
    .rename(columns={'sum':'fraud_count','count':'total'})
    .assign(fraud_rate=lambda x: x['fraud_count'] / x['total'] * 100)
    .sort_values('fraud_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 7))
median_rate = cat_stats['fraud_rate'].median()
colors = [PALETTE_FRAUD if r > median_rate else PALETTE_LEGIT
          for r in cat_stats['fraud_rate']]
bars = ax.barh(cat_stats.index, cat_stats['fraud_rate'], color=colors, edgecolor='white')
ax.set_title('Fraud Rate by Merchant Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Fraud Rate (%)')
ax.axvline(median_rate, color='#64748b', linestyle='--', linewidth=1.2,
           label=f'Median = {median_rate:.2f}%')
ax.legend(fontsize=9)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height()/2,
            f'{w:.2f}%', va='center', fontsize=8)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
savefig(fig, '04_fraud_by_category.png')
plt.show()

## Plot 5 · Feature Correlation Heatmap

In [ ]:
corr = X_train.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.4, linecolor='#e2e8f0',
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
savefig(fig, '05_correlation_heatmap.png')
plt.show()

## Plot 6 · Feature Importance (Best Model)

In [ ]:
clf = model.named_steps['clf'] if hasattr(model, 'named_steps') else model
imp = clf.feature_importances_ if hasattr(clf, 'feature_importances_') else np.abs(clf.coef_[0])

imp_df = pd.DataFrame({'feature': feature_names, 'importance': imp}).sort_values('importance')
colors = [PALETTE_FRAUD if v >= np.percentile(imp_df['importance'], 75)
          else PALETTE_LEGIT for v in imp_df['importance']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(imp_df['feature'], imp_df['importance'], color=colors, edgecolor='white')
ax.set_title(f'Feature Importance — {type(clf).__name__}', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score (Gini)')
for i, val in enumerate(imp_df['importance']):
    ax.text(val + imp_df['importance'].max()*0.005, i, f'{val:.4f}', va='center', fontsize=8)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
savefig(fig, '06_feature_importance.png')
plt.show()

## Plot 7 · Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ['Legitimate', 'Fraud']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrix — Best Model', fontsize=14, fontweight='bold')

for ax, (norm, title) in zip(axes, [(None, 'Raw Counts'), ('true', 'Normalised (row %)') ]):
    if norm == 'true':
        cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt, vmax = '.2%', 1.0
    else:
        cm_plot, fmt, vmax = cm, ',', None
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=1, linecolor='white', ax=ax,
                vmin=0, vmax=vmax,
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

fig.tight_layout()
savefig(fig, '07_confusion_matrix.png')
plt.show()

## Plot 8 · ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val  = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.fill_between(fpr, tpr, alpha=0.12, color=PALETTE_FRAUD)
ax.plot(fpr, tpr, color=PALETTE_FRAUD, linewidth=2.5,
        label=f'Random Forest  (AUC = {roc_auc_val:.4f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1.2, label='Random Classifier (AUC = 0.50)')
ax.set_title('ROC Curve — Best Model', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(loc='lower right', fontsize=10)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
savefig(fig, '08_roc_curve.png')
plt.show()

## Plot 9 · Precision-Recall Curve & Threshold Analysis

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
pr_auc_val = average_precision_score(y_test, y_prob)
baseline   = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Precision-Recall Analysis — Best Model', fontsize=14, fontweight='bold')

ax = axes[0]
ax.fill_between(recall, precision, alpha=0.12, color=PALETTE_FRAUD)
ax.plot(recall, precision, color=PALETTE_FRAUD, linewidth=2.5,
        label=f'Random Forest  (AP = {pr_auc_val:.4f})')
ax.axhline(baseline, color='#64748b', linestyle='--', linewidth=1.2,
           label=f'Baseline (fraud rate = {baseline:.2%})')
ax.set_title('Precision-Recall Curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.plot(thresholds, precision[:-1], color=PALETTE_LEGIT,  linewidth=2, label='Precision')
ax2.plot(thresholds, recall[:-1],    color=PALETTE_FRAUD,  linewidth=2, label='Recall')
f1 = 2*(precision[:-1]*recall[:-1])/(precision[:-1]+recall[:-1]+1e-9)
ax2.plot(thresholds, f1, color=PALETTE_ACCENT, linewidth=2, linestyle='--', label='F1-Score')
best_idx = np.argmax(f1)
ax2.axvline(thresholds[best_idx], color='#94a3b8', linewidth=1, linestyle=':')
ax2.text(thresholds[best_idx]+0.01, 0.05,
         f'Best F1 @ threshold={thresholds[best_idx]:.2f}', fontsize=8, color='#475569')
ax2.set_title('Precision, Recall & F1 vs Decision Threshold')
ax2.set_xlabel('Decision Threshold')
ax2.set_ylabel('Score')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=9)

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
savefig(fig, '09_precision_recall_curve.png')
plt.show()

## Plot 10 · Fraud Probability Distribution

In [ ]:
df_prob = pd.DataFrame({'probability': y_prob, 'label': y_test.values})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Predicted Fraud Probability Distribution', fontsize=14, fontweight='bold')

ax = axes[0]
for label, color, name in [(0, PALETTE_LEGIT, 'Legitimate'), (1, PALETTE_FRAUD, 'Fraud')]:
    sub = df_prob[df_prob['label'] == label]['probability']
    ax.hist(sub, bins=60, color=color, alpha=0.45, label=name, density=True, edgecolor='none')
    sub.plot.kde(ax=ax, color=color, linewidth=2)
ax.axvline(0.5, color='#64748b', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
ax.set_title('Probability Density by True Label')
ax.set_xlabel('Predicted Fraud Probability')
ax.set_ylabel('Density')
ax.set_xlim(0, 1)
ax.legend(fontsize=9)

ax2 = axes[1]
bins = np.linspace(0, 1, 41)
for label, color, name in [(0, PALETTE_LEGIT, 'Legitimate'), (1, PALETTE_FRAUD, 'Fraud')]:
    sub = df_prob[df_prob['label'] == label]['probability']
    ax2.hist(sub, bins=bins, color=color, alpha=0.65, label=name, edgecolor='none')
ax2.axvline(0.5, color='#64748b', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
ax2.set_title('Transaction Count by Predicted Probability')
ax2.set_xlabel('Predicted Fraud Probability')
ax2.set_ylabel('Count')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.legend(fontsize=9)

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
savefig(fig, '10_probability_distribution.png')
plt.show()

## ✅ Done!

In [ ]:
print('All 10 plots saved to outputs/visualizations/')
print('Files:')
for f in sorted(os.listdir(OUT_DIR)):
    print(f'  {f}')